# Proyek 2M Sembako: Data Science untuk Ongkir, Upah Kurir, & Jarak Berisiko

## 1. Identifikasi Masalah UMKM & Peran Data
* **Masalah Bisnis:** 2M Sembako harus menetapkan ongkir dan upah kurir yang memenuhi tiga syarat sekaligus: ongkir tidak melebihi GoSend/GrabExpress, kurir bermotor sendiri mendapat penghasilan setara UMK, dan 2M tidak menalangi kerugian. Dengan skema upah saat ini (Rp3.500 + Rp1.500/km), 2M mulai rugi pada jarak $\ge$ 2,25 km dan kurir baru mencapai ±30% target UMK.
* **Peran Data:** Menyusun pipeline wrangling yang bisa dijalankan ulang, menguji model biaya tim (Excel) di Python, menjawab pertanyaan bisnis, lalu menyerahkan dataset bersih ke tim Gen AI dan Full-Stack.
* **Status Data:** Log order bersifat **sintetis** (produk belum launch) dan berlabel `SINTETIS_SIMULASI`. Log ini dipakai untuk menguji pipeline, bukan untuk menarik kesimpulan bisnis. Analisis Q1–Q5 berbasis asumsi tim.
## 2. Pertanyaan Bisnis
1. Pada jarak berapa 2M mulai rugi bila ongkir dibatasi tarif pembanding dan upah memakai skema sekarang?
2. Apakah skema upah sekarang membuat kurir mencapai penghasilan setara UMK? Jika tidak, berapa kesenjangannya?
3. Variabel apa yang paling menggeser ongkir minimum dan penghasilan kurir?
4. Seberapa kokoh kesimpulan Q1 terhadap asumsi tarif pembanding yang belum terverifikasi?
5. Opsi kebijakan mana yang menutup kesenjangan, dan apa konsekuensinya?
## 3. Draf Ringkas README (Handover untuk Tim Gen AI & Full-Stack)
- **Masalah:** Menyediakan dataset bersih dan model biaya yang bisa dipakai ulang untuk pricing engine (Full-Stack) dan eksperimen model (Gen AI).
- **Fitur/Target:** Kandidat target `waktu_per_antaran_menit` (regresi) berdasarkan `jarak_rute_km`, `is_macet`, `jam`, `hari_minggu`, `klaster_motor`, dan `berat_total_kg`. Pembagian train/test menurut waktu (80/20).
- **Preprocessing:** Pembersihan duplikat, penyeragaman kategori, perbaikan jarak dan waktu, serta imputasi berat yang ditandai. Kolom keuangan (`untung_*`, `kantong_*`) **jangan** dipakai sebagai fitur karena bocor dari target.
- **Peringatan:** Model yang dilatih pada data sintetis hanya mempelajari ulang aturan pembuat data. Wajib dilatih ulang pada log PWA Kurir asli (detail di `docs/03`).
## 4. Proses Data Wrangling (Gathering, Assessing, Cleaning, & Export)
Tahapan ini membaca data mentah, mendeteksi masalah kualitasnya, membersihkannya dengan aturan yang tercatat, lalu mengekspor hasilnya ke `data/clean/`. Logikanya ada di `src/`; notebook ini memanggilnya dan menampilkan hasil langkah demi langkah.

Import Library

In [2]:
# Library standar
import sys, io, contextlib
from pathlib import Path

# Tentukan akar proyek supaya modul di src/ bisa di-import
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Modul proyek (logika ada di src/, notebook hanya memanggil)
from src.config import RAW, CLEAN, OUT_T, LABEL_SINTETIS
from src.model_biaya import load_params, load_motor, grid, hitung
from src import s02_assessing_cleaning as s02
from src import s04_analisis as s04
from src import s05_cek_akurasi as s05

# Tampilkan grafik langsung di notebook
%matplotlib inline

# Atur gaya grafik dan tampilan tabel
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 10})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# Muat parameter model dan master motor (dari Excel tim)
P, M = load_params(), load_motor()

# Palet warna dan format Rp yang sama dengan laporan
C, rupiah = s04.C, s04.rupiah
print("Akar proyek:", ROOT.name)

ModuleNotFoundError: No module named 'src'

1. GATHERING: Membaca Data Mentah

In [ ]:
# Daftar semua CSV mentah di data/raw beserta ukurannya
inventaris = pd.DataFrame(
    [(p.name, *pd.read_csv(p).shape) for p in sorted(RAW.glob("*.csv"))],
    columns=["file_csv_mentah", "baris", "kolom"])
inventaris

* **Sumber:** `Draft_DS_Kurir_v2.xlsx` (model biaya tim) dipindahkan ke CSV oleh `s01_gathering`, dan log order sintetis dibuat oleh `synth_orders` dengan seed tetap.
* **Catatan:** `order_log_sintetis_kebenaran_uji.csv` adalah kunci jawaban yang hanya dipakai di bagian akurasi. Pada data asli file ini tidak ada.

In [ ]:
# Baca log order mentah yang akan dibersihkan
raw = pd.read_csv(RAW / "order_log_sintetis_raw.csv")
print("Ukuran:", raw.shape, "| label sumber_data:", raw["sumber_data"].unique().tolist())
raw.head()

2. ASSESSING: Memeriksa Kualitas Data Mentah

In [ ]:
# Profil kolom: tipe data, jumlah kosong, persen kosong, nilai unik
s02.profil(raw)

Profil menunjukkan `ongkir_rp` dan `pembanding_rp` bertipe teks (seharusnya angka), `ditugaskan_at` bertipe teks, dan ada kolom yang kosong sebagian (`klaster_motor`, `jarak_rute_km`, `berat_total_kg`). Berikut jumlah masalah per jenis, dihitung langsung dari data mentah dan terpisah dari kode pembersihan.

In [3]:
# Hitung masalah selain duplikat pada baris yang sudah tidak ganda
r1 = raw.drop_duplicates()

# Versi rapi dari kode motor dan kurir_id sebagai pembanding
kode_rapi = r1["kode_motor"].astype(str).str.upper().str.replace(r"\s+", "", regex=True)
kurir_rapi = r1["kurir_id"].astype(str).str.upper().str.replace("-", "", regex=False)

# Baris yang waktu tiba-nya lebih awal dari berangkat (tertukar)
swap = pd.to_datetime(r1["tiba_at"], errors="coerce") < pd.to_datetime(r1["berangkat_at"], errors="coerce")

# Tabel jumlah masalah per jenis
assess = pd.DataFrame([
    ("Duplikat",   "baris duplikat persis",                  raw.duplicated().sum(),                                       len(raw)),
    ("Kategori",   "kode_motor tidak rapi (spasi/huruf kecil)", (r1["kode_motor"] != kode_rapi).sum(),                     len(r1)),
    ("Kategori",   "kurir_id format beda (k-07 vs K07)",      (r1["kurir_id"] != kurir_rapi).sum(),                         len(r1)),
    ("Kategori",   "label status_order tidak seragam",        (~r1["status_order"].isin(["berhasil", "gagal_pelanggan_tidak_ada"])).sum(), len(r1)),
    ("Kategori",   "klaster_motor kosong",                    r1["klaster_motor"].isna().sum(),                             len(r1)),
    ("Tipe data",  "ongkir_rp berupa teks 'Rp x.xxx'",        r1["ongkir_rp"].astype(str).str.contains("Rp").sum(),         len(r1)),
    ("Tipe data",  "pembanding_rp berupa teks 'Rp x.xxx'",    r1["pembanding_rp"].astype(str).str.contains("Rp").sum(),     len(r1)),
    ("Waktu",      "format tanggal campuran (dd/mm/yyyy)",    r1["ditugaskan_at"].astype(str).str.contains("/").sum(),      len(r1)),
    ("Waktu",      "berangkat_at dan tiba_at tertukar",       swap.sum(),                                                   len(r1)),
    ("Jarak",      "jarak_rute_km kosong",                    r1["jarak_rute_km"].isna().sum(),                             len(r1)),
    ("Jarak",      f"jarak <= 0 atau > {s02.MAX_RADIUS_KM:g} km (salah ketik)", ((r1["jarak_rute_km"] <= 0) | (r1["jarak_rute_km"] > s02.MAX_RADIUS_KM)).sum(), len(r1)),
    ("Berat",      "berat_total_kg kosong",                   r1["berat_total_kg"].isna().sum(),                            len(r1)),
], columns=["jenis", "masalah", "baris", "dari_total"])

# Persentase masalah terhadap total baris
assess["persen"] = (assess["baris"] / assess["dari_total"] * 100).round(1)
assess

NameError: name 'raw' is not defined

In [ ]:
# Grafik batang: masalah diurutkan dari yang paling sedikit ke paling banyak
a = assess.sort_values("baris")
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.barh(a["masalah"], a["baris"], color=C["oranye"])
for y, v in enumerate(a["baris"]):
    ax.text(v + 15, y, str(v), va="center", fontsize=8)
ax.tick_params(axis="y", labelsize=8)
ax.set(title=f"Masalah kualitas pada log order mentah ({len(raw):,} baris)".replace(",", "."), xlabel="Baris terdampak")
plt.tight_layout()
plt.show()

* **Temuan:** `status_order` paling berantakan (±77% baris, mis. `Berhasil`, `BERHASIL`, `selesai`), disusul format tanggal campuran (24%), `ongkir_rp`/`pembanding_rp` bertipe teks (15%), dan `klaster_motor` kosong (10%). Nilai kosong pada `berat_total_kg` (6%) dan `jarak_rute_km` (4%) lebih sedikit.
* **Paling berisiko:** `jarak_rute_km` (dasar upah per km) dan urutan waktu (dasar waktu per antaran). Keduanya bisa diperbaiki dan dicek silang di tahap cleaning.

3. CLEANING: Membersihkan Data (Sebelum & Sesudah)

`s02.bersihkan(raw, P, M)` mengembalikan tabel bersih dan **log pembersihan** (satu baris per aturan beserta jumlah baris terdampak).
* **Dipulihkan:** nilai yang bisa dipastikan diperbaiki, mis. jarak diturunkan dari snapshot upah (`upah_total = dasar + per_km × jarak`).
* **Ditandai:** nilai yang tidak bisa dipastikan diberi penanda, bukan diubah diam-diam (`jarak_diperbaiki`, `ts_diperbaiki`, `berat_estimasi`, `flag_waktu_aneh`, `bayar_tidak_sesuai`).
* **Penjaga:** aturan bertanda "harus 0" adalah alarm; bila tidak nol, ada yang perlu ditelusuri manual.

In [ ]:
# Jalankan pembersihan: hasilnya tabel bersih dan log per aturan
bersih, log = s02.bersihkan(raw, P, M)
print("Mentah:", raw.shape, "-> Bersih:", bersih.shape)
log

Pemeriksaan silang antara hitungan assessing dan log pembersihan:

In [ ]:
# Pasangan nama masalah: tabel assessing -> log pembersihan
pasangan = {
    "baris duplikat persis": "baris duplikat persis",
    "kode_motor tidak rapi (spasi/huruf kecil)": "kode_motor tidak rapi (spasi/huruf kecil)",
    "kurir_id format beda (k-07 vs K07)": "kurir_id format beda (k-07 vs K07)",
    "label status_order tidak seragam": "label status_order tidak seragam",
    "klaster_motor kosong": "klaster_motor kosong di log",
    "ongkir_rp berupa teks 'Rp x.xxx'": "ongkir_rp berupa teks 'Rp x.xxx'",
    "pembanding_rp berupa teks 'Rp x.xxx'": "pembanding_rp berupa teks 'Rp x.xxx'",
    "format tanggal campuran (dd/mm/yyyy)": "format tanggal campuran (dd/mm/yyyy)",
    "berangkat_at dan tiba_at tertukar": "berangkat_at & tiba_at tertukar",
    "jarak_rute_km kosong": "jarak_rute_km kosong",
    "berat_total_kg kosong": "berat_total_kg kosong",
}

# Bandingkan jumlah dari assessing dengan jumlah di log
dari_log = log.set_index("masalah")["baris_terdampak"]
silang = assess.set_index("masalah").loc[list(pasangan), ["baris"]].copy()
silang["di_log_pembersihan"] = [int(dari_log[v]) for v in pasangan.values()]
silang["cocok"] = silang["baris"] == silang["di_log_pembersihan"]
silang

Contoh nyata sebelum dan sesudah untuk tiap aturan (dicocokkan lewat `order_id`):

In [ ]:
# Samakan baris dan urutan data mentah dengan tabel bersih
r1 = raw.drop_duplicates().set_index("order_id")
b = bersih.set_index("order_id")
r1 = r1.loc[b.index]

# Tampilkan nilai apa adanya; repr membuat spasi di ujung teks ikut terlihat
def tampil_nilai(v):
    return "kosong (NaN)" if pd.isna(v) else repr(v)

# Ambil n contoh order yang terkena suatu aturan, lengkap dengan nilai sebelum dan sesudah
def contoh(aturan, kolom, mask, n=3):
    idx = mask[mask].index[:n]
    return pd.DataFrame({"aturan": aturan, "kolom": kolom, "order_id": idx,
                         "sebelum": r1.loc[idx, kolom].map(tampil_nilai).values,
                         "sesudah": b.loc[idx, kolom].map(tampil_nilai).values})

# Kumpulkan contoh dari semua aturan
contoh_semua = pd.concat([
    contoh("2 kode motor dirapikan",          "kode_motor",    r1["kode_motor"] != b["kode_motor"]),
    contoh("2 format kurir_id diseragamkan",  "kurir_id",      r1["kurir_id"] != b["kurir_id"]),
    contoh("2 label status diseragamkan",     "status_order",  r1["status_order"] != b["status_order"]),
    contoh("2 klaster diisi dari master T2",  "klaster_motor", r1["klaster_motor"].isna()),
    contoh("3 rupiah teks -> angka",          "ongkir_rp",     r1["ongkir_rp"].astype(str).str.contains("Rp")),
    contoh("3 rupiah teks -> angka",          "pembanding_rp", r1["pembanding_rp"].astype(str).str.contains("Rp")),
    contoh("4 format tanggal campuran",       "ditugaskan_at", r1["ditugaskan_at"].astype(str).str.contains("/")),
    contoh("4 waktu tertukar dikembalikan",   "berangkat_at",  b["ts_diperbaiki"]),
    contoh("5 jarak dipulihkan dari upah",    "jarak_rute_km", b["jarak_diperbaiki"]),
    contoh("6 berat diimputasi median",       "berat_total_kg", b["berat_estimasi"]),
], ignore_index=True)
contoh_semua

Aturan duplikat (60 baris) tidak muncul di tabel contoh karena barisnya dibuang, terlihat dari jumlah baris `3060 → 3000`. Untuk jarak dan berat, sebaran sebelum dan sesudah lebih informatif:

In [ ]:
# Histogram jarak (sebelum, skala log), jarak (sesudah), dan berat (sebelum vs sesudah imputasi)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
ax[0].hist(r1["jarak_rute_km"].dropna(), bins=40, color=C["oranye"])
ax[0].set(title="Jarak sebelum (skala log; ada salah ketik)", xlabel="km", ylabel="order", yscale="log")
ax[1].hist(b["jarak_rute_km"], bins=40, color=C["hijau"])
ax[1].set(title="Jarak sesudah", xlabel="km")
ax[2].hist(r1["berat_total_kg"].dropna(), bins=30, alpha=.6, color=C["oranye"], label="sebelum (tanpa yang kosong)")
ax[2].hist(b["berat_total_kg"], bins=30, alpha=.6, color=C["hijau"], label="sesudah (imputasi median)")
ax[2].set(title="Berat: spike di median = imputasi", xlabel="kg")
ax[2].legend(fontsize=7)
plt.tight_layout()
plt.show()

* **Jarak:** sebelum cleaning ada nilai negatif dan outlier sampai puluhan km; sesudahnya semua berada di rentang wajar (0–8 km).
* **Berat:** batang tinggi di sekitar median adalah baris imputasi. Baris ini ditandai `berat_estimasi = True` dan tidak boleh dipakai untuk menilai pengaruh berat.

Aturan "harus 0" dan beberapa jaminan skema diuji dengan `assert`. Bila ada yang gagal, sel berhenti dengan galat, bukan lolos diam-diam.

In [ ]:
# Uji hasil pembersihan: ID unik, timestamp utuh dan berurutan, jarak valid, kategori dan label benar
assert bersih["order_id"].is_unique
assert bersih[s02.TS].notna().all().all()
assert (bersih[s02.TS].diff(axis=1).iloc[:, 1:] >= pd.Timedelta(0)).all().all()
assert (bersih["jarak_rute_km"] > 0).all() and bersih["jarak_rute_km"].max() <= s02.MAX_RADIUS_KM
assert bersih["klaster_motor"].notna().all() and bersih["berat_total_kg"].notna().all()
assert set(bersih["status_order"]) <= {"berhasil", "gagal_pelanggan_tidak_ada"}
assert (bersih["sumber_data"] == LABEL_SINTETIS).all()
print("Semua pemeriksaan lolos:", len(bersih), "baris, tidak ada duplikat, jarak/waktu valid, label sintetis utuh.")

4. EXPORT UNTUK TIM GEN AI & FULL-STACK

In [ ]:
# Simpan lewat fungsi resmi pipeline (s02.main) agar file di data/clean sama dengan hasil run_all.py
with contextlib.redirect_stdout(io.StringIO()):
    s02.main()

# Baca ulang file yang tersimpan dan cocokkan dengan hasil di memori
tersimpan = pd.read_csv(CLEAN / "order_log_bersih.csv")
sama = tersimpan.shape == bersih.shape and (tersimpan["order_id"].values == bersih["order_id"].values).all()
print("Tersimpan:", tersimpan.shape, "| konsisten dengan hasil di memori:", sama)

# Dataset fitur-target dengan pembagian train/test menurut waktu (80/20)
fitur = s02.dataset_fitur_target(bersih)
print("\nDataset fitur-target (untuk tim Gen AI), split menurut waktu 80/20:")
print(fitur["split"].value_counts().to_string())
print("\nFile di data/clean:", ", ".join(sorted(p.name for p in CLEAN.glob("*.csv"))))

## 5. Analisis Pertanyaan Bisnis (Q1–Q5)
Dasarnya model biaya `src/model_biaya.py`, hasil menulis ulang Excel tim ke Python yang sudah direkonsiliasi (bagian 6). `s04.run()` menghitung semua jawaban, menyimpan tabel ke `output/tabel/` dan grafik ke `output/grafik/`, lalu mengembalikan angka kunci `K` beserta tabelnya. Grafik di bawah digambar ulang dari tabel itu agar tampil di samping penjelasan.

**Ingat:** ini analisis skenario berbasis asumsi tim, bukan hasil transaksi nyata.

In [ ]:
# Jalankan seluruh analisis: angka kunci K, tabel amplop upah, Q2, Q5, dan batching
K, amp, q2, q5, bat = s04.run()
print("Angka kunci tersedia:", len(K), "butir (juga tersimpan di output/tabel/angka_kunci.json)")

Q1. Pada jarak berapa 2M mulai rugi?
(skema sekarang: upah Rp3.500 + Rp1.500/km, ongkir dibatasi tarif pembanding, klaster A)

In [ ]:
# Grid jarak 0,25-10 km untuk klaster A
gA = grid(P, M, "A")

# Ringkasan: jarak mulai rugi dan kantong 2M di beberapa titik jarak
print(f"2M mulai rugi pada jarak {K['q1_rugi_mulai_km']:.2f} km; {K['q1_titik_rugi_skema_baru']} dari {len(gA)} titik grid (0,25-10 km) rugi.")
print("Kurir sendiri rugi di titik mana pun?", bool((gA["untung_kurir_per_antaran"] <= 0).any()))
for d in (3, 5, 8, 10):
    print(f"  jarak {d:>2} km: kantong 2M per antaran = {rupiah(K[f'q1_kantong_{d}km'])}")

In [ ]:
# Kiri: kantong 2M per antaran vs jarak. Kanan: ongkir minimum vs pembanding
x = gA["jarak_rute_km"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(x, gA["kantong_2m"], color=C["merah"], lw=2, label="Skema baru (ongkir <= pembanding)")
ax[0].plot(x, gA["kantong_2m_lama"], color=C["abu"], lw=1.5, ls="--", label="Tarif lama (Rp3.000/km, min Rp8.000)")
ax[0].axhline(0, color="k", lw=.8)
ax[0].fill_between(x, gA["kantong_2m"], 0, where=gA["kantong_2m"] < 0, color=C["merah"], alpha=.12)
ax[0].axvline(K["q1_rugi_mulai_km"], color=C["merah"], ls=":", lw=1)
ax[0].annotate(f"2M mulai rugi\n{K['q1_rugi_mulai_km']:.2f} km", (K["q1_rugi_mulai_km"], 0), (3.3, 2200),
               arrowprops=dict(arrowstyle="->"), color=C["merah"])
ax[0].set(title="Kantong 2M per antaran vs jarak (klaster A)", xlabel="Jarak rute satu arah (km)", ylabel="Rp per antaran")
ax[0].legend(fontsize=8, loc="lower left")
ax[1].plot(x, gA["ongkir_minimum_2m"], color=C["merah"], lw=2, label="Ongkir minimum agar 2M tidak rugi")
ax[1].plot(x, gA["pembanding"], color=C["biru"], lw=2, label="Pembanding GoSend (turunan data PIC)")
ax[1].plot(x, gA["ongkir_lama"], color=C["abu"], ls="--", label="Ongkir tarif lama")
ax[1].set(title="Ongkir minimum vs pembanding", xlabel="Jarak rute satu arah (km)", ylabel="Rp")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Amplop upah** adalah upah maksimum per antaran yang masih membuat 2M tidak rugi bila ongkir sama dengan pembanding:

In [ ]:
# Tabel amplop upah per jarak, dibandingkan dengan upah skema sekarang
amp

**Kesimpulan Q1:**
* 2M mulai rugi pada jarak 2,25 km dan terus rugi sampai 10 km. Kurir sendiri tidak pernah rugi.
* **Penyebab:** upah naik Rp1.500 per km, sedangkan tarif pembanding (menurut asumsi saat ini) datar di tarif minimum sampai ±4 km. Skema sekarang melewati amplop upah mulai 3 km.
* **Efek samping:** pada upah amplop, untung kurir per jam turun tajam di jarak jauh (kolom `untung_kurir_per_jam_di_upah_maks`), sehingga antaran jauh berisiko ditolak.
* Titik 2,25 km bergantung pada asumsi tarif pembanding (lihat Q4).

Q2. Apakah kurir mencapai penghasilan setara UMK?

In [ ]:
# Ringkasan penghasilan kurir pada skenario Dasar dan kebutuhan ongkir untuk mencapai target
print(f"Skenario Dasar, klaster A: penghasilan bersih {rupiah(K['q2_dasarA_penghasilan'])}/bulan = {K['q2_dasarA_persen_umk']:.1%} dari target UMK (klaster B: {K['q2_dasarB_persen_umk']:.1%}).")
print(f"Untung kurir {rupiah(K['q2_dasarA_per_jam'])}/jam vs target {rupiah(K['q2_target_per_jam'])}/jam.")
print(f"Antaran/hari dibutuhkan untuk 100% target: {K['q2_antaran_dibutuhkan_dasarA']}  | kapasitas waktu: {K['q2_kapasitas_per_hari']}")
print(f"Untuk capai target di jarak 2,7 km lewat ongkir: {rupiah(K['q2_ongkir_target_2_7km'])} (pembanding {rupiah(K['q2_pembanding_2_7km'])}); selisih {rupiah(K['q2_gap_per_order'])}/order = {rupiah(K['q2_gap_per_kurir_bulan'])} per kurir per bulan.")

In [ ]:
# Tabel per skenario; hanya kolom uang yang dibulatkan, jarak_km tetap apa adanya
tampil = q2.assign(persen_umk=(q2["persen_umk"] * 100).round(1))
tampil = tampil.round({"untung_per_antaran": 0, "penghasilan_bulan": 0, "untung_per_jam": 0})
tampil

In [ ]:
# Batang: penghasilan bersih kurir sebagai persen target UMK per skenario dan klaster
fig, ax = plt.subplots(figsize=(8.5, 4))
lab = [f"{r.skenario}\n(klaster {r.klaster})" for r in q2.itertuples()]
col = [C["biru"] if r.klaster == "A" else C["ungu"] for r in q2.itertuples()]
ax.bar(range(len(q2)), q2["persen_umk"] * 100, color=col)
ax.axhline(100, color=C["merah"], ls="--")
ax.text(len(q2) - .5, 102, "target = 100% UMK", ha="right", color=C["merah"], fontsize=8)
for i, v in enumerate(q2["persen_umk"] * 100):
    ax.text(i, v + 1.5, f"{v:.0f}%", ha="center", fontsize=8)
ax.set_xticks(range(len(q2)))
ax.set_xticklabels(lab, fontsize=7)
ax.set(ylim=(0, 115), ylabel="% dari target penghasilan bersih (UMK Kab. Bekasi)", title="Penghasilan bersih kurir per bulan vs target UMK")
plt.tight_layout()
plt.show()

**Kesimpulan Q2:**
* Pada semua skenario (optimis sampai pesimis, klaster A dan B) penghasilan kurir jauh di bawah target; yang tertinggi hanya ±34%.
* **Pembatas utama:** kapasitas waktu. Pada skenario Dasar kurir sanggup ±15 antaran per hari, sedangkan target menuntut 50.
* Menutup kesenjangan lewat ongkir tidak realistis (ongkir yang dibutuhkan beberapa kali lipat pembanding). Target "100% UMK" perlu ditinjau ulang atau dicapai lewat tuas lain (Q5).

Q3. Variabel apa yang paling menggeser ongkir minimum dan penghasilan kurir?
(tornado ±20%, nilai dari Excel tim, divalidasi ulang di bagian 6)

In [ ]:
# Baca tabel tornado dan tampilkan variabel paling berpengaruh
tor = pd.read_csv(OUT_T / "q3_tornado.csv")
print("Paling menggeser ongkir minimum  :", K["q3_top_ongkir"])
print("Paling menggeser penghasilan kurir:", K["q3_top_hasil"])
tor.round(0)

In [ ]:
# Tornado: kiri untuk ongkir minimum, kanan untuk penghasilan kurir per bulan
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
t1, t2 = tor.sort_values("swing_ongkir"), tor.sort_values("swing_hasil")
for a, t, lo, hi, ttl in ((ax[0], t1, "d_ongkir_minus", "d_ongkir_plus", "Perubahan ongkir minimum (Rp)"),
                          (ax[1], t2, "d_hasil_minus", "d_hasil_plus", "Perubahan penghasilan kurir/bulan (Rp)")):
    a.barh(t["variabel"], t[lo], color=C["biru"], label="variabel turun / skenario Dekat")
    a.barh(t["variabel"], t[hi], color=C["oranye"], label="variabel naik / skenario Jauh")
    a.axvline(0, color="k", lw=.8)
    a.set_title(ttl, fontsize=10)
    a.tick_params(axis="y", labelsize=8)
ax[0].legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.show()

**Kesimpulan Q3:**
* **Ongkir minimum** paling dipengaruhi sebaran jarak order, lalu upah per km dan upah dasar.
* Harga bensin dan efisiensi motor **tidak** menggeser ongkir minimum karena biaya motor ditanggung kurir, bukan 2M. Keduanya hanya menggeser penghasilan kurir.
* **Penghasilan kurir** paling peka terhadap upah per km, lalu upah dasar, disusul throughput (antaran per hari). Tuas yang efektif ada di skema upah dan jumlah antaran, bukan di harga BBM.

Q4. Seberapa kokoh kesimpulan Q1 terhadap asumsi tarif pembanding?
(tarif pembanding diturunkan dari rentang harga Rp7.500–15.500 dengan dua asumsi yang belum terverifikasi: jarak yang tercakup tarif minimum, dan jarak saat harga maksimum berlaku; Q1 dihitung ulang pada 25 kombinasi)

In [4]:
# Baca tabel 25 kombinasi; ">10" dianggap NaN = tidak rugi sampai 10 km
rob = pd.read_csv(OUT_T / "q4_robustness_asumsi_pembanding.csv", index_col=0)
num = rob.apply(pd.to_numeric, errors="coerce")
print(f"Jarak mulai rugi berkisar {K['q4_rugi_range'][0]:.2f} - {K['q4_rugi_range'][1]:.2f} km; {K['q4_n_tidak_rugi']} dari 25 kombinasi tidak rugi sampai 10 km.")
rob

NameError: name 'OUT_T' is not defined

In [ ]:
# Heatmap jarak mulai rugi menurut kombinasi asumsi tarif pembanding
fig, ax = plt.subplots(figsize=(6.2, 4))
ax.imshow(num.fillna(11).values, cmap="RdYlGn", vmin=1, vmax=8, origin="lower")
for i in range(num.shape[0]):
    for j in range(num.shape[1]):
        v = num.iloc[i, j]
        ax.text(j, i, ">10" if np.isnan(v) else f"{v:.2f}", ha="center", va="center", fontsize=9)
ax.set_xticks(range(num.shape[1])); ax.set_xticklabels(num.columns)
ax.set_yticks(range(num.shape[0])); ax.set_yticklabels(num.index)
ax.set(xlabel="Jarak (km) pada harga maks Rp15.500", ylabel="Jarak (km) yang tercakup harga min Rp7.500",
       title="Jarak mulai 2M rugi (km), klaster A\nmenurut asumsi tarif pembanding")
ax.grid(False)
plt.tight_layout()
plt.show()

**Kesimpulan Q4:**
* Bila tarif minimum mencakup **≥ 3 km**, titik rugi tetap 2,25 km di semua nilai jarak-harga-maksimum. Kesimpulan Q1 kokoh.
* Bila tarif minimum hanya mencakup 1–2 km, titik rugi bergeser sampai 5,75 km, dan pada 2 kombinasi 2M tidak rugi sampai 10 km.
* **Tindak lanjut:** satu kali simulasi order di aplikasi pada jarak 1, 3, 5, dan 8 km akan menyelesaikan ketidakpastian ini.

Q5. Opsi kebijakan mana yang menutup kesenjangan, dan apa konsekuensinya?
(sebaran jarak "Dasar" 40/40/20 untuk 0–2, 2–4, >4 km, klaster A)
* **O0** skema sekarang · **O1** upah dibatasi amplop · **O2** radius layanan ≤ 2 km · **O3** ongkir fleksibel di atas pembanding · **O4** subsidi 2M menutup selisih

In [ ]:
# Tabel perbandingan opsi; nama opsi dipendekkan menjadi O0-O4
q5_tampil = q5.assign(opsi=q5["opsi"].str.slice(0, 2)).round(3)
q5_tampil[["opsi", "order_terlayani", "kantong_2m_rata2", "porsi_order_2m_rugi", "subsidi_2m_per_order",
           "penghasilan_kurir_bulan", "persen_umk", "catatan"]]

In [ ]:
# Batang opsi kebijakan; O4 diberi warna oranye karena selisihnya ditutup anggaran subsidi
nm = [o[:2] for o in q5["opsi"]]
warna = [C["hijau"] if v >= 0 else C["merah"] for v in q5["kantong_2m_rata2"]]
warna[-1] = C["oranye"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
ax[0].barh(nm, q5["kantong_2m_rata2"], color=warna)
ax[0].axvline(0, color="k", lw=.8)
ax[0].set(title="Kantong 2M rata-rata per order, sebelum subsidi (Rp)")
ax[0].invert_yaxis()
ax[1].barh(nm, q5["persen_umk"] * 100, color=C["biru"])
ax[1].axvline(100, color=C["merah"], ls="--")
ax[1].set(title="Penghasilan bersih kurir (% UMK)", xlim=(0, 105))
plt.tight_layout()
plt.show()

**Batching (ilustratif):** mengantar beberapa order dalam satu perjalanan. Parameter detour (0,5 km) dan waktu tambahan (8 menit per drop) adalah **asumsi**, bukan data.

In [ ]:
# Ringkasan batching: untung kurir per jam vs target
print(f"Target untung kurir: {rupiah(K['q2_target_per_jam'])}/jam | 1 order/trip: {rupiah(K['q6_per_jam_k1'])}/jam | 3 order/trip: {rupiah(K['q6_per_jam_k3'])}/jam")
print("Jumlah order per trip yang mencapai target (sampai 8):", K["q6_k_capai_target"])

# Garis untung kurir per jam menurut jumlah order per trip, dengan garis target
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(bat["order_per_trip"], bat["untung_kurir_per_jam"], marker="o", color=C["biru"], label="Untung kurir per jam")
ax.axhline(K["q2_target_per_jam"], color=C["merah"], ls="--", label="Target per jam")
ax.set(xlabel="Order per trip", ylabel="Rp per jam", title="Batching: untung kurir per jam (3 km, upah amplop)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Kesimpulan Q5:**
* **O0** (skema sekarang): rugi untuk 60% order pada sebaran Dasar.
* **O1:** 2M aman, tetapi penghasilan kurir turun dan antaran jauh jadi tidak menarik.
* **O2:** 2M aman, tetapi hanya ±40% order terlayani.
* **O3:** 2M aman dan penghasilan kurir tetap, dengan harga ongkir jarak jauh di atas pembanding. Respons pelanggan belum teruji.
* **O4:** ongkir dan upah tidak berubah; selisih ditutup subsidi ±Rp1.114 per order. Penghasilan kurir tetap 30,4% UMK, dan batas subsidi belum bisa diputuskan sebelum contribution margin per order diketahui.
* **Batching:** tuas terbesar untuk penghasilan kurir per jam, tetapi sendirian masih di bawah target.
* Tidak ada opsi tunggal yang menjaga 2M aman, ongkir ≤ pembanding, *dan* kurir mendekati UMK. Rekomendasi lengkap untuk PM ada di `docs/02_laporan_analisis_dan_rekomendasi.md`.

## 6. Pemeriksaan Akurasi
`s05_cek_akurasi` menjalankan lima kelompok pemeriksaan dan mencatat hasilnya ke `output/tabel/cek_akurasi.csv`:
* **A. Rekonsiliasi:** apakah model Python menghasilkan angka yang sama dengan nilai di Excel tim (T4, Output_Tidy, T5, T6, tornado)?
* **B. Invarian:** apakah identitas logika terjaga (kantong = ongkir − biaya, ongkir ≤ batas, monotonik, proporsi 100%)?
* **C. Pembersihan:** seberapa tepat pembersihan memulihkan nilai sebenarnya? (hanya bisa karena data sintetis punya kunci jawaban)
* **D. Data publik:** data eksternal mana yang terverifikasi, mana yang masih asumsi?
* **E. Audit workbook:** apa yang janggal di workbook Excel tim?

In [ ]:
# Kosongkan hasil global s05 supaya Run All berulang tidak menggandakan pemeriksaan
s05.HASIL.clear()

# Jalankan semua pemeriksaan tanpa mencetak log panjang
with contextlib.redirect_stdout(io.StringIO()):
    cek = s05.main()

# Ringkasan jumlah PASS/INFO/TEMUAN per kelompok
ringkas = cek.groupby(["kelompok", "hasil"]).size().unstack(fill_value=0)
ringkas.loc["TOTAL"] = ringkas.sum()
ringkas

In [ ]:
# Total hasil, jumlah FAIL (harus 0), dan daftar yang bukan PASS
print("Total pemeriksaan:", len(cek), "|", cek["hasil"].value_counts().to_dict())
print("FAIL (harus 0):", int((cek["hasil"] == "FAIL").sum()))
cek[cek["hasil"] != "PASS"][["id", "hasil", "deskripsi", "nilai"]]

**Kesimpulan akurasi:**
* **Tidak ada FAIL.** Model Python cocok dengan Excel (selisih ≤ 1e-6 pada grid dan Output_Tidy; ≤ 1e-3 pada T5, T6, tornado), jadi menulis ulang model ke Python tidak mengubah hasil.
* **INFO** adalah angka yang dilaporkan tanpa ambang lulus-gagal, mis. galat imputasi berat dan jumlah data eksternal yang terverifikasi.
* **TEMUAN** adalah catatan audit workbook, bukan kegagalan kode. Yang terpenting untuk keputusan: parameter `jarak_tarif_maks_km` masih asumsi dan hasil sangat sensitif terhadapnya (Q4).
* **Kelompok C** membuktikan pipeline berfungsi pada data sintetis, **bukan** jaminan kualitas pada data asli. Saat log PWA Kurir tersedia, kunci jawaban tidak ada; gantikan dengan pemeriksaan sampel manual.

Status data eksternal dan asumsi (kelompok D):

In [ ]:
# Tabel status verifikasi data eksternal dan tindak lanjutnya
pd.read_csv(OUT_T / "cek_akurasi_data_publik.csv")[["data", "nilai_dipakai", "status_verifikasi", "tindak_lanjut"]]

## 7. Kesimpulan & Langkah Berikutnya
* **Sudah kuat:** pipeline bisa dijalankan ulang dari nol dengan hasil identik, model sudah direkonsiliasi dengan Excel, dan setiap perbaikan data tercatat di log pembersihan.
* **Belum bisa diklaim:** semua kesimpulan bisnis masih berbasis asumsi tim dan data sintetis.
* **Validasi berikutnya:** (1) tarif GoSend/GrabExpress di aplikasi pada beberapa jarak, (2) log order asli dari PWA Kurir, (3) contribution margin per order untuk menentukan batas subsidi.
* **Dokumen terkait:** `docs/02_laporan_analisis_dan_rekomendasi.md`, `docs/03_panduan_dataset_GenAI_dan_FullStack.md`, `docs/04_pemeriksaan_akurasi.md`, `docs/05_kebutuhan_dan_keputusan.md`.